In [2]:
import pandas as pd
import json


In [3]:
# load samples data

filename = "TCGA-BRCA.star_tpm"
df = pd.read_csv(f'../data/{filename}.csv')
df = df.drop(columns=['Unnamed: 0'])
# df = df.drop(columns=['Unnamed: 0.1', 'Unnamed: 0'])
df.head()

,TSPAN6,TNMD,DPM1,SCYL3,C1orf112,FGR,CFH,FUCA2,GCLC,NFYA,...,H3C2,H3C3,TMX2-CTNND1,C3orf36,C8orf44,C8orf44-SGK3,NPBWR1,CDR1,ACTL10,PANO1
0,5.662037,3.376096,6.860140,4.400552,2.845169,3.579965,4.996181,5.614683,4.385652,5.843705,...,0.472176,0.183582,0.158725,1.368154,2.452964,0.000000,0.170310,0.0,0.0,1.122739
1,3.703721,0.463099,7.086452,4.051007,2.426989,2.322361,4.943860,6.539078,4.046404,5.711726,...,1.557631,1.015712,0.056306,0.456491,1.756596,0.029842,0.062398,0.0,0.0,1.243182
2,6.514515,0.000000,6.805072,5.037264,4.043248,2.124130,2.993348,5.119912,3.409907,5.963742,...,0.238787,0.130404,0.253021,0.607863,2.212818,0.000000,0.015640,0.0,0.0,0.604261
3,4.784917,2.328061,6.443071,4.374970,4.162790,3.122375,4.754674,5.608738,3.625691,5.678199,...,1.242755,1.054918,0.026729,0.904735,2.209079,0.028003,0.039419,0.0,0.0,0.536351
4,4.251984,0.435415,6.178436,3.558464,2.589548,1.508581,3.722149,5.224044,3.242221,5.956145,...,0.000000,0.000000,0.206518,0.417704,2.761541,0.000000,1.522257,0.0,0.0,0.389126


In [4]:
# make a list of genes out of columns

genes = pd.Index(df.columns)
genes

Index(['TSPAN6', 'TNMD', 'DPM1', 'SCYL3', 'C1orf112', 'FGR', 'CFH', 'FUCA2',
       'GCLC', 'NFYA',
       ...
       'H3C2', 'H3C3', 'TMX2-CTNND1', 'C3orf36', 'C8orf44', 'C8orf44-SGK3',
       'NPBWR1', 'CDR1', 'ACTL10', 'PANO1'],
      dtype='object', length=20560)

## Match on gene names

In [5]:
# load gene info data, drop duplacates

gene_info_original = pd.read_csv('../data/gene_info_table.csv')
gene_info = gene_info_original.drop(columns=['ensembl_id', 'Unnamed: 0'])
gene_info = gene_info.drop_duplicates()
gene_info.head()

,gene_name,gene_type
0,TSPAN6,protein_coding
1,TNMD,protein_coding
2,DPM1,protein_coding
3,SCYL3,protein_coding
4,C1orf112,protein_coding


In [6]:
# choose 1 gene_type for ambiguous genes

gene_info["priority"] = (gene_info["gene_type"] == "protein_coding").astype(int)
gene_info = gene_info.sort_values(
    ["gene_name", "priority"],
    ascending=[True, False]
)
gene_info = gene_info.drop_duplicates("gene_name")
gene_info = gene_info.drop(columns="priority")


In [7]:
gene_type_map = gene_info.set_index("gene_name")["gene_type"]
mapped_types = genes.map(gene_type_map)
mapped_types

Index(['protein_coding', 'protein_coding', 'protein_coding', 'protein_coding',
       'protein_coding', 'protein_coding', 'protein_coding', 'protein_coding',
       'protein_coding', 'protein_coding',
       ...
       'protein_coding', 'protein_coding', 'protein_coding', 'protein_coding',
       'protein_coding', 'protein_coding', 'protein_coding', 'protein_coding',
       'protein_coding',              nan],
      dtype='object', length=20560)

In [8]:
type_counts = mapped_types.value_counts(dropna=False)
print(type_counts)


protein_coding                        17273
NaN                                    1318
antisense                               602
lncRNA                                  499
lincRNA                                 456
IG_V_gene                               124
TR_V_gene                               101
processed_transcript                     56
IG_V_pseudogene                          46
pseudogene                               20
TR_V_pseudogene                          17
polymorphic_pseudogene                   14
IG_C_gene                                13
IG_C_pseudogene                           7
TR_C_gene                                 4
transcribed_unprocessed_pseudogene        4
IG_J_gene                                 2
transcribed_unitary_pseudogene            1
sense_intronic                            1
misc_RNA                                  1
miRNA                                     1
Name: count, dtype: int64


In [9]:
n_total = len(genes)
n_matched = mapped_types.notna().sum()
n_missing = mapped_types.isna().sum()

print("Total genes:", n_total)
print("Matched:", n_matched)
print("Missing:", n_missing)
print("Match %:", n_matched / n_total * 100)


Total genes: 20560
Matched: 19242
Missing: 1318
Match %: 93.58949416342412


In [10]:
missing_genes = genes[mapped_types.isna()]
print(missing_genes.tolist())


['VPS50', 'TSPOAP1', 'REX1BD', 'KDM7A', 'EEF1AKNMT', 'ELOA', 'ADGRA2', 'MRE11', 'RTF2', 'SARS1', 'EIPR1', 'ADSS2', 'MTREX', 'RIPOR1', 'SLC66A1', 'RIPOR3', 'JADE2', 'NEXMIF', 'HPF1', 'GATB', 'GUCY1B1', 'WAPL', 'BICRA', 'INTS13', 'CCN5', 'BORCS8-MEF2B', 'KARS1', 'PACC1', 'PLPP1', 'ADGRF5', 'HDHD5', 'UFD1', 'ELP1', 'CFAP20', 'BUD23', 'ADGRL1', 'SELENOO', 'NCBP3', 'HACD3', 'MYDGF', 'JADE1', 'TIGAR', 'NRDC', 'CARMIL1', 'PUM3', 'DELE1', 'FYB1', 'DOP1A', 'TUT7', 'FAM234B', 'AK6', 'RTRAF', 'CFAP61', 'GCN1', 'P3H2', 'AARS1', 'ADA2', 'WHRN', 'SMIM24', 'MACROH2A2', 'ATP5F1D', 'CBARP', 'MARCHF2', 'ESS2', 'GRK3', 'SNU13', 'SEPTIN3', 'RSPH14', 'CCDC198', 'SUSD6', 'PCNX1', 'PPP4R3A', 'PRORP', 'RAB5IF', 'PRELID3B', 'PAK5', 'H2BW2', 'RUBCNL', 'ACOD1', 'CIAO3', 'ANTKMT', 'ELOB', 'VPS35L', 'CCN4', 'PYCR3', 'CGB3', 'TLE5', 'YJU2', 'DMAC2', 'NOP53', 'PLPPR2', 'CFAP69', 'GSDME', 'H2AZ2', 'GARS1', 'MINDY4', 'RIC1', 'VSIR', 'TWNK', 'EDRF1', 'NEURL1', 'STN1', 'TASOR2', 'CRACD', 'NSD2', 'ZPR1', 'JHY', 'KMT5B', 

## Using another table for mapping to ensemble id

In [11]:
# another gene_info for missed genes

gene_ensembl = pd.read_csv('../data/gene_info.csv')
gene_ensembl.head()

,Unnamed: 0,soma_joinid,feature_id,feature_name,feature_length
0,0,0,ENSG00000121410,A1BG,3999
1,1,1,ENSG00000268895,A1BG-AS1,3374
2,2,2,ENSG00000148584,A1CF,9603
3,3,3,ENSG00000175899,A2M,6318
4,4,4,ENSG00000245105,A2M-AS1,2948


In [12]:
symbol_to_ens_map = gene_ensembl.set_index("feature_name")["feature_id"]
missing_ens = missing_genes.map(symbol_to_ens_map)


# genes_ens = pd.Index(missing_genes)
# ensembl_ids = genes_ens.map(symbol_to_ens)

# genes_ens[ensembl_ids.isna()] # check if all are mapped
missing_ens.notna().mean()



1.0

In [13]:
gene_info_ens = gene_info_original.drop(columns=['gene_name', 'Unnamed: 0'])
gene_info_ens = gene_info_ens.drop_duplicates()
gene_info_ens.head()

,ensembl_id,gene_type
0,ENSG00000000003,protein_coding
1,ENSG00000000005,protein_coding
2,ENSG00000000419,protein_coding
3,ENSG00000000457,protein_coding
4,ENSG00000000460,protein_coding


In [14]:
# choose 1 gene_type for ambiguous genes

gene_info_ens["priority"] = (gene_info_ens["gene_type"] == "protein_coding").astype(int)
gene_info_ens = gene_info_ens.sort_values(
    ["ensembl_id", "priority"],
    ascending=[True, False]
)
gene_info_ens = gene_info_ens.drop_duplicates("ensembl_id")
gene_info_ens = gene_info_ens.drop(columns="priority")

# creat mapping
gene_type_map_ens = gene_info_ens.set_index("ensembl_id")["gene_type"]


In [15]:
missing_types_from_ens = missing_ens.map(gene_type_map_ens)


In [16]:
# fill in missing genes with new mapping
mapped_types = pd.Series(mapped_types, index=genes)
mapped_types.loc[missing_genes] = missing_types_from_ens.values

mapped_types


TSPAN6          protein_coding
TNMD            protein_coding
DPM1            protein_coding
SCYL3           protein_coding
C1orf112        protein_coding
                     ...      
C8orf44-SGK3    protein_coding
NPBWR1          protein_coding
CDR1            protein_coding
ACTL10          protein_coding
PANO1                      NaN
Length: 20560, dtype: object

In [17]:
# see how many genes are in categories
mapped_types.value_counts(dropna=False)

protein_coding                        18234
antisense                               714
lincRNA                                 659
lncRNA                                  523
IG_V_gene                               124
TR_V_gene                               101
processed_transcript                     62
IG_V_pseudogene                          46
pseudogene                               28
TR_V_pseudogene                          17
polymorphic_pseudogene                   14
IG_C_gene                                13
IG_C_pseudogene                           7
TR_C_gene                                 5
transcribed_unprocessed_pseudogene        4
IG_J_gene                                 2
unprocessed_pseudogene                    2
transcribed_unitary_pseudogene            1
sense_intronic                            1
misc_RNA                                  1
miRNA                                     1
NaN                                       1
Name: count, dtype: int64

## Choose only protein-coding genes

In [18]:
protein_coding_genes = mapped_types[mapped_types == "protein_coding"].index.tolist()
len(protein_coding_genes)

18234

In [19]:
with open(f"../data/{filename}_gene_list.json", "w") as f:
    json.dump(protein_coding_genes, f)